# Dictionary lookup

Employee and product notes go into a dictionary. If a question word matches a name, that note is pasted into the system prompt.


## Client


In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)
sys.path.insert(0, str(repo_root / "W5-RAG"))

from store import CHAT_MODEL, knowledge_base

from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
model = CHAT_MODEL
print(model)


gpt-4o-mini


## Knowledge


In [2]:
knowledge = {}

for path in sorted((knowledge_base / "employees").glob("*.md")):
    knowledge[path.stem.split(" ")[-1].lower()] = path.read_text(encoding="utf-8")

for path in sorted((knowledge_base / "products").glob("*.md")):
    knowledge[path.stem.lower()] = path.read_text(encoding="utf-8")

print(len(knowledge))
print(sorted(knowledge))


38
['adams', 'anderson', 'bishop', 'bizllm', 'blake', 'brooks', 'carllm', 'carter', 'chen', 'claimllm', 'foster', 'greene', 'harper', 'healthllm', 'homellm', 'johnson', 'kim', 'lancaster', 'lifellm', 'liu', 'markellm', 'martinez', "o'brien", 'park', 'patel', 'rellm', 'rivera', 'rodriguez', 'sharma', 'spencer', 'thompson', 'thomson', 'tran', 'trenton', 'walker', 'williams', 'wilson', 'zhang']


## Match words


In [3]:
def get_relevant_context(message):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]


def additional_context(message):
    relevant = get_relevant_context(message)
    if not relevant:
        return "There is no additional context relevant to the user's question."
    return (
        "The following additional context might be relevant in answering the user's question:\n\n"
        + "\n\n".join(relevant)
    )


matched = get_relevant_context("Who is Lancaster and what is carllm?")
print(len(matched), "notes")
print(matched[0][:400])
print("---")
print(matched[1][:400])


2 notes
# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insuranc
---
# Product Summary

# Carllm

## Summary

Carllm is an innovative auto insurance product developed by Insurellm, designed to streamline the way insurance companies offer coverage to their customers. Powered by cutting-edge artificial intelligence, Carllm utilizes advanced algorithms to deliver personalized auto insurance solutions, ensuring optimal coverage while minimizing costs. With a robust inf


## Chat


In [4]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""


def chat(message, history):
    messages = [{"role": "system", "content": SYSTEM_PREFIX + additional_context(message)}]
    messages += history
    messages.append({"role": "user", "content": message})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        timeout=120.0,
    )
    return response.choices[0].message.content


print(chat("Who is Lancaster?", []))
print()
print(chat("What is carllm?", []))


Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, an Insurance Tech company based in San Francisco, California. She co-founded the company in 2015 and has played a significant role in its growth and innovation in the insurance market.



Carllm is an innovative auto insurance product developed by Insurellm that utilizes artificial intelligence to streamline the way insurance companies offer coverage. It provides personalized auto insurance solutions, enhances customer satisfaction, and minimizes costs through features like AI-powered risk assessment, instant quoting, customizable coverage plans, fraud detection, and a customer insights dashboard. Carllm is designed for both B2B and B2C customers and aims to transform the auto insurance landscape.


## Page


In [5]:
import gradio as gr

demo = gr.ChatInterface(
    chat,
    title="Insurellm",
    examples=["Who is Lancaster?", "What is carllm?"],
)


def launch():
    user = os.getenv("GRADIO_USER")
    password = os.getenv("GRADIO_PASSWORD")
    auth = (user, password) if user and password else None
    demo.queue()
    demo.launch(auth=auth)


In [6]:
launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
